# Kharita : parcours en six tâches

Les valeurs manquantes sont conservées, jamais transformées automatiquement en zéro. Les modèles utilisent une séparation des communes et les caractéristiques antérieures à la cible.

## Machine learning : régression et clustering

In [ ]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"Data/resultats-elections-communales-maroc-2015-2021.csv").exists())
sys.path.insert(0,str(ROOT/"Script"))
from kharita_pipeline import load_raw,clean_data,make_comparison,build_features,save_clean_outputs
import pandas as pd,numpy as np,matplotlib.pyplot as plt,seaborn as sns
sns.set_theme(style="whitegrid")
raw=load_raw(); df,quality,party_cols=clean_data(raw)
print(f"{len(df):,} lignes; années {quality['years']}")

## Machine learning : régression et clustering

Le notebook suit les choix de nettoyage documentés; les champs absents restent manquants et les analyses restent descriptives.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor,HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
features=build_features(df,party_cols); data=features.dropna(subset=["turnout_2021"]); y=data.turnout_2021; X=data.drop(columns="turnout_2021")
cat=["region"]; num=[c for c in X if c not in cat]; pre=ColumnTransformer([("num",make_pipeline(SimpleImputer(strategy="median"),StandardScaler()),num),("cat",make_pipeline(SimpleImputer(strategy="most_frequent"),OneHotEncoder(handle_unknown="ignore")),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42)
models={"Linear Regression":LinearRegression(),"Random Forest":RandomForestRegressor(n_estimators=250,min_samples_leaf=2,random_state=42,n_jobs=-1),"Gradient Boosting":GradientBoostingRegressor(random_state=42),"HistGradientBoosting":HistGradientBoostingRegressor(random_state=42)}
rows=[]
for name,est in models.items():
 pipe=make_pipeline(pre,est); pipe.fit(Xtr,ytr); pred=pipe.predict(Xte); rows.append({"modèle":name,"MAE":mean_absolute_error(yte,pred),"RMSE":mean_squared_error(yte,pred)**.5,"R2":r2_score(yte,pred)})
display(pd.DataFrame(rows).sort_values("MAE"))
from sklearn.cluster import KMeans,AgglomerativeClustering
from sklearn.metrics import silhouette_score,davies_bouldin_score
w=make_comparison(df); cols=["turnout_pct_2015","turnout_pct_2021","variation_points","party_concentration_2015","winning_margin_2015"]; cl=w[cols].dropna(); Z=StandardScaler().fit_transform(cl); scores=[]
for k in range(2,7):
 for name,labels in [("K-Means",KMeans(n_clusters=k,n_init=20,random_state=42).fit_predict(Z)),("Agglomerative",AgglomerativeClustering(n_clusters=k).fit_predict(Z))]: scores.append({"algo":name,"k":k,"silhouette":silhouette_score(Z,labels),"davies_bouldin":davies_bouldin_score(Z,labels)})
display(pd.DataFrame(scores).sort_values("silhouette",ascending=False))